In [3]:
## Initialize Hyperparameters and import libraries

import numpy as np
import torch

from model import *
from utilities import *
from loss_ftn import *

In [4]:
PATH = "D:\\01_Datasets\\DIRTL_Manuscript\\"

data_dims = np.shape(np.load(PATH + "true_results_multiwl\\result_Hy_"+f'{0:08d}'+'k'+f"{0.01:.4f}"+".npy")[0])

In [5]:
n_train = 9000
n_test = 1000

batch_size = 250
epochs = 100

layer_num = 10

In [6]:
train_loader ,test_loader = data_loader(n_train, n_test, batch_size, wl_list, data_dims, 0, dz)

Loading Test Data: 100%|██████████| 1000/1000 [01:05<00:00, 15.36it/s]


In [7]:
model = FNOModel2d(modes=16, width=32, blocks=layer_num).cuda()

In [8]:
lr_top = 0.0005
step_size = 10
gamma = 0.5

In [9]:
name = (n_train/1000)
SAVE_model = f"DIRTL_{name:.2f}k_baseline_{layer_num}FL.pth"
SAVE_lc =  f"DIRTL_{name:.2f}k_baseline_{layer_num}FL_LC.npz"

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr_top)

Train_rel_L1_arr = []
Train_rel_L2_arr = []
Test_rel_L1_arr = []
Test_rel_L2_arr = []

# Define StepLR scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,    # number of epochs before decay
    gamma=gamma             # decay factor, e.g., 0.1 reduces LR by 10x
)

loss = nn.MSELoss()

# gc.collect(k)
torch.cuda.empty_cache()

total_time = 0

for ep in range(epochs):
    t1 = default_timer()
    model.train()
    Train_mse = 0

    for input_shape, result in train_loader:
        input_shape, result = input_shape.cuda(), result.cuda()
        optimizer.zero_grad()
        
        out = model((input_shape))

        Train_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))

        Train_mse_temp.backward()
        
        optimizer.step()
        
        Train_mse += Train_mse_temp.detach() * batch_size

    scheduler.step()

    model.eval()
    Test_mse = 0.0
    with torch.no_grad():
        for input_shape, result in test_loader:
            input_shape, result = input_shape.cuda(), result.cuda()

            out = model((input_shape))
            Test_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))
            Test_mse += Test_mse_temp.detach() * batch_size

    Train_mse /= len(train_loader.dataset)
    Test_mse /= len(test_loader.dataset)

    Train_rmse = np.sqrt(Train_mse.item())
    Test_rmse = np.sqrt(Test_mse.item())

    if SAVE_lc:
        with torch.no_grad():
            model.eval()
            _, _, rel_L1, rel_L2, _ = rel_err(model, train_loader)
            Train_rel_L1_arr.append(np.mean(rel_L1))
            Train_rel_L2_arr.append(np.mean(rel_L2))

            _, _, rel_L1, rel_L2, _ = rel_err(model, test_loader)
            Test_rel_L1_arr.append(np.mean(rel_L1))
            Test_rel_L2_arr.append(np.mean(rel_L2))
        
    t2 = default_timer()
    total_time += t2 - t1

    print(f"Epoch {ep+1}, Time: {t2-t1:.2f}s, Train RMSE: {Train_rmse:.4f}, Test RMSE: {Test_rmse:.4f}")

print(f"total time: {total_time:.2f}")

if SAVE_model:
    torch.save(model.state_dict(), SAVE_model)

if SAVE_lc:
    np.savez(SAVE_lc,
        Train_rel_L1=Train_rel_L1_arr,
        Train_rel_L2=Train_rel_L2_arr,
        Test_rel_L1=Test_rel_L1_arr,
        Test_rel_L2=Test_rel_L2_arr)

Epoch 1, Time: 147.21s, Train RMSE: 0.4683, Test RMSE: 0.3514
Epoch 2, Time: 144.08s, Train RMSE: 0.3576, Test RMSE: 0.3213
Epoch 3, Time: 140.52s, Train RMSE: 0.3412, Test RMSE: 0.2894
Epoch 4, Time: 142.81s, Train RMSE: 0.3333, Test RMSE: 0.3184
Epoch 5, Time: 141.92s, Train RMSE: 0.3217, Test RMSE: 0.2512
Epoch 6, Time: 141.01s, Train RMSE: 0.3197, Test RMSE: 0.2607
Epoch 7, Time: 143.07s, Train RMSE: 0.3110, Test RMSE: 0.2997
Epoch 8, Time: 143.29s, Train RMSE: 0.3128, Test RMSE: 0.2924
Epoch 9, Time: 142.19s, Train RMSE: 0.2991, Test RMSE: 0.2528
Epoch 10, Time: 140.54s, Train RMSE: 0.3039, Test RMSE: 0.3093
Epoch 11, Time: 139.20s, Train RMSE: 0.2644, Test RMSE: 0.2438
Epoch 12, Time: 138.72s, Train RMSE: 0.2510, Test RMSE: 0.2744
Epoch 13, Time: 138.19s, Train RMSE: 0.2655, Test RMSE: 0.1847
Epoch 14, Time: 139.54s, Train RMSE: 0.2555, Test RMSE: 0.2405
Epoch 15, Time: 138.05s, Train RMSE: 0.2549, Test RMSE: 0.2699
Epoch 16, Time: 140.12s, Train RMSE: 0.2460, Test RMSE: 0.1902
E